# Rhetorical oracle: inference only
Keep this notebook and outputs private. Attach the frozen corpus v1 `train.jsonl` and `validation.jsonl` split files; do not attach the combined corpus or test file. Enable the `HF_TOKEN` Secret and internet, select GPU T4 x2. Only GPU 0 is exposed.
This runs 63 greedy Gemma 3 4B generations: ordinary prompt, source plan, and reference-derived oracle plan, across the same 21 development examples. It performs no training. No reference wording is sent to generation. Oracle C is diagnostic, not deployable.
After generation, Gemma is unloaded and an independent NLI model runs on CPU with synthetic sanity controls. Automatic semantic scores remain proxies. Budget 63 generations plus download and CPU evaluation time; progress is printed every 30 seconds during slow operations.
Results: `/kaggle/working/rhetorical_oracle_v1/inference/evaluation.json`, `generations.private.jsonl`, `metrics.private.jsonl`, `nli_controls.json`, and `run_status.json`. Public code contains no private paragraphs.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"


In [ ]:
%pip install -q "transformers==5.20.1" "accelerate>=1.0" "bitsandbytes>=0.46" "sentencepiece>=0.2"


In [ ]:
import json, sys
from pathlib import Path
runtime = Path("/kaggle/working/_rhetorical_runtime/src")
files = json.loads('{"fama_style/__init__.py": "\\"\\"\\"Fama-style project modules; imports do not load model weights.\\"\\"\\"\\n", "fama_style/prompting/__init__.py": "\\"\\"\\"Fama-style prompting modules; imports do not load model weights.\\"\\"\\"\\n", "fama_style/prompting/corpus.py": "\\"\\"\\"Frozen multi-paper v1 instruction, identical to the curated data export.\\"\\"\\"\\n\\nFORMAT = \\"fama_multi_paper_v1\\"\\nINSTRUCTION = (\\n    \\"Rewrite the passage below in the formal academic economics style of Eugene Fama \\"\\n    \\"and his coauthors. Preserve every substantive claim, actor, attribution, \\"\\n    \\"condition, qualification, uncertainty, negation, quantity, comparison, and \\"\\n    \\"causal relationship. Do not add or omit claims or strengthen tentative claims. \\"\\n    \\"Return only one coherent paragraph, without headings, Markdown, explanations, \\"\\n    \\"or commentary.\\"\\n)\\n\\n\\ndef make_prompt(text):\\n    return f\\"{INSTRUCTION}\\\\n\\\\nPassage:\\\\n{text}\\"\\n", "fama_style/research/__init__.py": "\\"\\"\\"CPU-first claim-preserving rewriting research; no training launchers.\\"\\"\\"\\n", "fama_style/research/claims.py": "\\"\\"\\"Evidence-backed surface ledger and conservative rewrite verification.\\n\\nThis is a bounded English-rule prototype, not a semantic parser or NLI model.\\nUnrecognized paraphrases abstain; matching protected words alone never passes.\\n\\"\\"\\"\\nimport collections\\nimport hashlib\\nimport re\\n\\nTOKEN = re.compile(r\\"\\\\d+(?:\\\\.\\\\d+)?%?|[^\\\\W_]+(?:[\'’][^\\\\W_]+)?|[<>=+−-]\\", re.UNICODE)\\nGUARD = re.compile(r\\"\\\\b(if|unless|when|provided|assuming|because|therefore|thus|hence|not|no|without|may|might|could|would|should)\\\\b\\", re.I)\\nPATTERNS = {\\n    \\"quantities\\": r\\"\\\\b\\\\d+(?:[.,]\\\\d+)*(?:\\\\s*%)?\\",\\n    \\"negation\\": r\\"\\\\b(?:not|no|never|neither|nor|without)\\\\b\\",\\n    \\"modality\\": r\\"\\\\b(?:may|might|can|could|would|should|must|will|possible|possibly|expected|expect|expects)\\\\b\\",\\n    \\"scope\\": r\\"\\\\b(?:all|every|each|most|many|some|only|always|usually|generally|often|sometimes|exactly|at least|at most)\\\\b\\",\\n    \\"condition_cues\\": r\\"\\\\b(?:if|unless|when|provided that|assuming|as long as|conditional on)\\\\b\\",\\n    \\"causality\\": r\\"\\\\b(?:cause|causes|caused|lead to|leads to|because|therefore|thus|hence|implies|imply|increases|decreases|reduces|raises)\\\\b\\",\\n}\\n\\n\\ndef sha(text):\\n    return hashlib.sha256(text.encode()).hexdigest()\\n\\n\\ndef tokens(text):\\n    return TOKEN.findall(text.casefold())\\n\\n\\ndef sentences(text):\\n    return [part.strip() for part in re.split(r\\"(?<=[.!?])\\\\s+(?=[A-Z])\\", text.strip()) if part.strip()]\\n\\n\\ndef conditional(text):\\n    \\"\\"\\"Only recognize a single explicit if-condition with no nested guard.\\"\\"\\"\\n    body = text.rstrip(\\". \\")\\n    front = re.fullmatch(r\\"If ([^,;]+), ([^;]+)\\", body, re.I)\\n    back = re.fullmatch(r\\"([^,;]+) if ([^,;]+)\\", body, re.I)\\n    if front:\\n        condition, proposition = front.groups()\\n    elif back:\\n        proposition, condition = back.groups()\\n    else:\\n        return None\\n    if re.search(r\\"\\\\b(if|unless|only|because|then|otherwise)\\\\b\\", condition+\\" \\"+proposition, re.I):\\n        return None\\n    return {\\"condition\\": condition.strip(), \\"proposition\\": proposition.strip()}\\n\\n\\ndef ledger(text):\\n    claims = []\\n    for index, sentence in enumerate(sentences(text)):\\n        start = text.find(sentence, claims[-1][\\"evidence\\"][\\"end\\"] if claims else 0)\\n        protected = {key: [{\\"text\\": match.group(), \\"start\\": start+match.start(), \\"end\\": start+match.end()}\\n                           for match in re.finditer(pattern, sentence, re.I)] for key, pattern in PATTERNS.items()}\\n        condition = conditional(sentence)\\n        # Actor/predicate hypotheses retain exact evidence; unresolved syntax is explicit.\\n        actor = re.match(r\\"(.{1,100}?)\\\\s+(?:is|are|was|were|has|have|may|might|can|could|would|should|must|will|find|finds|use|uses|argue|argues|predict|predicts|reduce|reduces|increase|increases|cause|causes)\\\\b\\", sentence, re.I)\\n        actors = [actor.group(1)] if actor else []\\n        relations = []\\n        for cue in protected[\\"causality\\"]:\\n            offset = cue[\\"start\\"]-start\\n            left, right = sentence[:offset].strip(), sentence[offset+len(cue[\\"text\\"]):].strip()\\n            direction = \\"right_explains_left\\" if cue[\\"text\\"].lower() == \\"because\\" else \\"left_to_right_hypothesis\\"\\n            relations.append({\\"cue\\": cue[\\"text\\"], \\"left\\": left, \\"right\\": right,\\n                              \\"direction\\": direction, \\"verified_parse\\": False})\\n        claims.append({\\"claim_id\\": f\\"c{index+1}\\", \\"actors\\": actors,\\n                       \\"proposition\\": condition[\\"proposition\\"] if condition else sentence,\\n                       \\"conditions\\": ([{\\"text\\":condition[\\"condition\\"],\\"parse_status\\":\\"single_if_rule\\"}] if condition else\\n                                      [{\\"text\\":sentence,\\"cue\\":cue[\'text\'],\\"parse_status\\":\\"unresolved_scope\\"}\\n                                       for cue in protected[\'condition_cues\']]),\\n                       **protected, \\"causal_relations\\": relations,\\n                       \\"evidence\\": {\\"text\\": sentence, \\"start\\": start, \\"end\\": start+len(sentence)},\\n                       \\"actor_parse_status\\": \\"heuristic\\" if actor else \\"unresolved\\"})\\n    return {\\"source_sha256\\": sha(text), \\"claims\\": claims,\\n            \\"coverage\\": \\"All sentence spans retained; atomic claims, coreference and implicit relations are not fully parsed\\",\\n            \\"independently_semantically_verified\\": False}\\n\\n\\ndef invariants(text):\\n    return {key: collections.Counter(match.group().lower().replace(\\" \\", \\"\\")\\n                                     for match in re.finditer(pattern, text, re.I))\\n            for key, pattern in PATTERNS.items()}\\n\\n\\ndef risk_flags(source, target):\\n    a, b = invariants(source), invariants(target)\\n    return [key+\\"_surface_mismatch\\" for key in a if a[key] != b[key]]\\n\\n\\ndef variants(text):\\n    \\"\\"\\"Reproducible candidate grammar; no target/reference text is used.\\"\\"\\"\\n    result = [{\\"operation\\": \\"identity\\", \\"text\\": text}]\\n    parts = sentences(text)\\n    # Do not fuse conditional/negated/modal/causal clauses or quoted material.\\n    for i in range(len(parts)-1):\\n        left, right = parts[i:i+2]\\n        if GUARD.search(left+\\" \\"+right) or any(c in left+right for c in (\'\\"\', \'“\', \'”\')):\\n            continue\\n        if not left.endswith(\'.\') or re.search(r\\"(?:\\\\b[A-Z]|e\\\\.g|i\\\\.e|et al)\\\\.$\\", left):\\n            continue\\n        right=re.sub(r\\"^(The|This|These|Those|It|We|A|An)\\\\b\\",lambda match:match.group().lower(),right)\\n        joined = left[:-1]+\\"; \\"+right\\n        result.append({\\"operation\\": \\"join_independent_sentences\\", \\"text\\": \\" \\".join(parts[:i]+[joined]+parts[i+2:])})\\n        if len(result) >= 6:\\n            break\\n    for i, part in enumerate(parts):\\n        condition = conditional(part)\\n        if condition:\\n            c, p = condition[\\"condition\\"], condition[\\"proposition\\"]\\n            rewritten = p+\\" if \\"+c+\\".\\" if part.lower().startswith(\\"if \\") else \\"If \\"+c+\\", \\"+p+\\".\\"\\n            operation = \\"condition_to_end\\" if part.lower().startswith(\\"if \\") else \\"condition_to_front\\"\\n            result.append({\\"operation\\": operation, \\"text\\": \\" \\".join(parts[:i]+[rewritten]+parts[i+1:])})\\n        if \\";\\" in part and not GUARD.search(part) and not any(c in part for c in (\'\\"\',\'“\',\'”\')):\\n            pieces = part.split(\\";\\")\\n            if len(pieces) == 2 and all(piece.strip() for piece in pieces):\\n                result.append({\\"operation\\": \\"split_independent_clauses\\",\\n                               \\"text\\": \\" \\".join(parts[:i]+[pieces[0].strip()+\\". \\"+pieces[1].strip()[0].upper()+pieces[1].strip()[1:]]+parts[i+1:])})\\n    # Explicit author voice transformation is one-directional: do not guess that\\n    # an arbitrary \'the authors\' phrase refers to this paper rather than a citation.\\n    if re.search(r\\"\\\\bWe\\\\b\\", text) and not any(c in text for c in (\'\\"\',\'“\',\'”\')):\\n        result.append({\\"operation\\": \\"neutralize_author_voice\\", \\"text\\": re.sub(r\\"\\\\bWe\\\\b\\", \\"The authors\\", text)})\\n    unique = {}\\n    for candidate in result:\\n        unique.setdefault(candidate[\\"text\\"], candidate)\\n    return list(unique.values())\\n\\n\\ndef verify(source, candidate):\\n    \\"\\"\\"Pass only exact identity or a replayable conservative surface operation.\\"\\"\\"\\n    flags = risk_flags(source, candidate)\\n    if flags:\\n        return {\\"status\\": \\"rejected\\", \\"reasons\\": flags, \\"semantic_proof\\": False}\\n    replay = next((v for v in variants(source) if v[\\"text\\"] == candidate), None)\\n    if replay is not None:\\n        return {\\"status\\": \\"surface_preserved\\", \\"operation\\": replay[\\"operation\\"],\\n                \\"reasons\\": [], \\"semantic_proof\\": False,\\n                \\"scope\\": \\"Replayable bounded surface grammar; not full natural-language entailment\\"}\\n    return {\\"status\\": \\"unknown\\", \\"reasons\\": [\\"outside_verified_surface_grammar\\"], \\"semantic_proof\\": False}\\n", "fama_style/research/experiments.py": "\\"\\"\\"Prepare separate weakly supervised SFT/DPO research data; never trains.\\"\\"\\"\\nimport collections\\nimport re\\n\\nfrom .claims import sha, variants, verify, risk_flags\\nfrom .signal import overlap, features, distance\\n\\n\\ndef paraphrases(source, limit=4):\\n    \\"\\"\\"Two-hop bounded rewrites with replayable step-by-step provenance.\\"\\"\\"\\n    queue=[(source,[])];seen={source};outputs=[]\\n    for _ in range(2):\\n        next_queue=[]\\n        for text,path in queue:\\n            for candidate in variants(text):\\n                rewritten=candidate[\'text\']\\n                if rewritten in seen or verify(text,rewritten)[\'status\']!=\'surface_preserved\':\\n                    continue\\n                seen.add(rewritten)\\n                step={\\"operation\\":candidate[\'operation\'],\\"input_sha256\\":sha(text),\\"output_sha256\\":sha(rewritten),\\n                      \\"text\\":rewritten}\\n                chain=path+[step]\\n                outputs.append({\\"text\\":rewritten,\\"operations\\":chain,\\n                                \\"lexical_change\\":1-overlap(source,rewritten),\\"features\\":features(rewritten)})\\n                next_queue.append((rewritten,chain))\\n        queue=next_queue\\n    # Prefer a range of structural feature vectors; lexical-only micro-edits are reported.\\n    selected=[];feature_sets=set()\\n    for item in sorted(outputs,key=lambda item:-item[\'lexical_change\']):\\n        signature=tuple(round(v,3) for v in item[\'features\'].values())\\n        if signature not in feature_sets:\\n            selected.append(item);feature_sets.add(signature)\\n        if len(selected)>=limit:break\\n    return selected\\n\\n\\ndef corruption(source):\\n    replacements={\\"may\\":\\"will\\",\\"might\\":\\"will\\",\\"some\\":\\"all\\",\\"many\\":\\"all\\",\\"not\\":\\"\\"}\\n    for match in re.finditer(r\\"\\\\b(may|might|some|many|not)\\\\b\\",source,re.I):\\n        changed=source[:match.start()]+replacements[match.group().lower()]+source[match.end():]\\n        if risk_flags(source,changed):\\n            return {\\"text\\":changed,\\"operation\\":\\"deliberate_\\"+match.group().lower()+\\"_mutation\\",\\n                    \\"changed_span\\":[match.start(),match.end()]}\\n    return None\\n\\n\\ndef prepare(rows, profile):\\n    if any(row[\'split\']!=\'train\' for row in rows):raise ValueError(\'Training preparation requires training rows only\')\\n    sft,preferences,quarantine,coverage=[],[],[],[]\\n    for row in rows:\\n        source,target=row[\'plain_english\'],row[\'completion\']\\n        inherited=risk_flags(source,target)\\n        inputs=paraphrases(source)\\n        for variant in inputs:\\n            sft.append({\\"parent_id\\":row[\'id\'],\\"parent_paper\\":row[\'paper_key\'],\\"split\\":\\"train\\",\\n                        \\"input\\":variant[\'text\'],\\"target\\":target,\\"target_sha256\\":sha(target),\\n                        \\"original_input_sha256\\":sha(source),\\"provenance\\":variant,\\n                        \\"reference_alignment_flags\\":inherited,\\"semantic_status\\":\\"surface_equivalent_to_curated_input_only\\",\\n                        \\"training_ready\\":False})\\n        coverage.append({\\"id\\":row[\'id\'],\\"variants\\":len(inputs),\\n                         \\"lexically_diverse_variants\\":sum(item[\'lexical_change\']>=.1 for item in inputs),\\n                         \\"reference_alignment_flags\\":inherited})\\n        negatives=[]\\n        # Avoid automatically treating already-clear or low-contrast input as bad.\\n        contrast=distance(source,profile)-distance(target,profile)\\n        if overlap(source,target)<.8 and contrast>.1:\\n            negatives.append({\\"category\\":\\"copying\\",\\"text\\":source,\\"contrast_proxy\\":contrast})\\n        for candidate in variants(target):\\n            if candidate[\'operation\'] in (\'neutralize_author_voice\',\'split_independent_clauses\'):\\n                if verify(target,candidate[\'text\'])[\'status\']==\'surface_preserved\' and distance(candidate[\'text\'],profile)-distance(target,profile)>.05:\\n                    negatives.append({\\"category\\":\\"generic_surface_proxy\\",\\"text\\":candidate[\'text\'],\\n                                      \\"operation\\":candidate[\'operation\']})\\n        bad=corruption(source)\\n        if bad:negatives.append({\\"category\\":\\"semantic_error\\",\\"text\\":bad[\'text\'],\\"mutation\\":bad})\\n        for negative in negatives:\\n            item={\\"parent_id\\":row[\'id\'],\\"split\\":\\"train\\",\\"prompt\\":row.get(\'prompt\',source),\\n                  \\"chosen\\":target,\\"rejected\\":negative[\'text\'],\\"negative_category\\":negative[\'category\'],\\n                  \\"chosen_sha256\\":sha(target),\\"original_input_sha256\\":sha(source),\\"provenance\\":negative,\\n                  \\"reference_alignment_flags\\":inherited,\\"training_ready\\":False,\\n                  \\"label_status\\":\\"weak_preference_not_independent_semantic_verification\\"}\\n            if inherited:quarantine.append(item)\\n            else:preferences.append(item)\\n    summary={\\"training_parent_rows\\":len(rows),\\"sft_variants_staged\\":len(sft),\\n             \\"parents_with_two_or_more_variants\\":sum(item[\'variants\']>=2 for item in coverage),\\n             \\"parents_with_two_lexically_diverse_variants\\":sum(item[\'lexically_diverse_variants\']>=2 for item in coverage),\\n             \\"weak_preferences\\":len(preferences),\\"quarantined_preferences\\":len(quarantine),\\n             \\"weak_preference_categories\\":dict(collections.Counter(row[\'negative_category\'] for row in preferences)),\\n             \\"all_preference_categories\\":dict(collections.Counter(row[\'negative_category\'] for row in preferences+quarantine)),\\n             \\"training_ready\\":False,\\n             \\"reason\\":\\"Bounded surface paraphrases are not a rich semantic-invariance generator; reference alignment and style proxies need automated model-based calibration before training.\\",\\n             \\"generic_negative_caveat\\":\\"Third-person/clause-splitting controls proxy generic surface style, not the entire space of fluent generic prose.\\"}\\n    return {\\"sft\\":sft,\\"preferences\\":preferences,\\"quarantined_preferences\\":quarantine,\\"coverage\\":coverage,\\"summary\\":summary}\\n\\n\\ndef plans():\\n    common={\\"status\\":\\"prepared_not_executed\\",\\"gpu_submission_allowed\\":False,\\"model_id\\":\\"google/gemma-3-4b-it\\",\\n            \\"quantization\\":\\"NF4 FP32\\",\\"microbatch_size\\":1,\\"gradient_accumulation_steps\\":2,\\n            \\"sequence_limit\\":1024,\\"truncate\\":False,\\"lora_rank\\":8,\\"language_qv_only\\":True,\\n            \\"reserved_test_use\\":False,\\"human_review_required\\":False}\\n    return {\\"A_semantic_invariant_sft\\":{**common,\\"hypothesis\\":\\"Diverse content-equivalent inputs reduce the shortcut of copying one neutral phrasing.\\",\\n                \\"target\\":\\"unchanged original training reference\\",\\"maximum_paraphrases_per_parent\\":4,\\n                \\"sample_weight\\":\\"equal parent weight, not equal weight per generated row\\",\\n                \\"next_gate\\":\\"automatically verified diversity/coverage and no target leakage; three-example five-update memory probe only after later authorization\\"},\\n            \\"B_contrastive_dpo\\":{**common,\\"hypothesis\\":\\"Explicit faithful-style versus copy/generic/error contrasts change the generation preference, beyond target likelihood.\\",\\n                \\"reference_policy\\":\\"frozen pilot/SFT snapshot; never silently adapter-disabled raw base when SFT reference is intended\\",\\n                \\"inspected_trl_commit\\":\\"88365e546b5be84821f1a8bd9f167ef3230950e2\\",\\n                \\"peft_reference_plan\\":\\"Use the inspected implementation\'s frozen adapter named ref, or precomputed reference scores fingerprinted to the exact SFT snapshot; do not pass obsolete ref_adapter_name arguments\\",\\n                \\"unvalidated_compatibility_gates\\":[\\"Gemma text path accepts fused_lm_head=True and returns log_probs\\", \\"ref adapter frozen while policy adapter updates\\", \\"cached reference scores equal uncached scores\\", \\"chosen and rejected jointly fit one T4 without truncation\\"],\\n                \\"trl_proposed_options\\":{\\"loss_type\\":[\\"sigmoid\\"],\\"beta\\":.1,\\"learning_rate\\":5e-6,\\n                    \\"per_device_train_batch_size\\":1,\\"precompute_ref_log_probs\\":True,\\"precompute_ref_batch_size\\":1,\\n                    \\"max_length\\":None,\\"padding_free\\":False,\\"sync_ref_model\\":False},\\n                \\"compatibility_status\\":\\"current upstream API inspected; not installed or GPU validated\\",\\n                \\"next_gate\\":\\"cache reference log-probs against the exact frozen SFT adapter, explicit no-truncation audit; then separately authorized five-pair memory probe\\"}}\\n", "fama_style/research/prototype.py": "\\"\\"\\"Claim ledger → empirical plan → candidates → conservative automated checks.\\"\\"\\"\\nfrom .claims import ledger, variants, verify, sha\\nfrom .signal import distance, overlap\\n\\n\\ndef rewrite(text, profile):\\n    claim_ledger=ledger(text)\\n    plan={\\"fit_split\\":\\"train\\",\\"operations\\":profile[\'eligible_operations\'],\\n          \\"source_distance\\":distance(text,profile),\\"reference_features\\":profile[\'features\']}\\n    candidates=[]\\n    for candidate in variants(text):\\n        verification=verify(text,candidate[\'text\'])\\n        gain=plan[\'source_distance\']-distance(candidate[\'text\'],profile)\\n        similarity=overlap(text,candidate[\'text\'])\\n        admissible=(verification[\'status\']==\'surface_preserved\' and candidate[\'operation\'] in plan[\'operations\']\\n                    and gain>.05 and similarity<.9)\\n        candidates.append({**candidate,\\"verification\\":verification,\\"style_gain_proxy\\":gain,\\n                           \\"copy_similarity\\":similarity,\\"eligible_for_selection\\":admissible})\\n    good=[candidate for candidate in candidates if candidate[\'eligible_for_selection\']]\\n    selected=max(good,key=lambda item:item[\'style_gain_proxy\']) if good else candidates[0]\\n    return {\\"source_sha256\\":sha(text),\\"claim_ledger\\":claim_ledger,\\"style_plan\\":plan,\\"candidates\\":candidates,\\n            \\"selected_text\\":selected[\'text\'],\\"status\\":\\"selected_bounded_rewrite\\" if good else \\"abstained_unchanged\\",\\n            \\"semantic_guarantee\\":False,\\"scope\\":\\"CPU rule prototype; no LLM, no trained model, no independent entailment validation\\"}\\n", "fama_style/research/run_extension.py": "\\"\\"\\"CPU-only signal diagnosis, constrained inference and training-data preparation.\\"\\"\\"\\nimport argparse\\nimport collections\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\n\\nfrom .claims import sha, risk_flags\\nfrom .signal import fit_profile, diagnose, overlap, distance\\nfrom .prototype import rewrite\\nfrom .experiments import prepare, plans\\n\\nROOT=Path(__file__).resolve().parents[3]\\nEXPECTED={\\"train\\":\\"a108416c1935615a7e01bb4d3fa86e0818b961bfb16f15d3883e6905654ee248\\",\\n          \\"validation\\":\\"f1c75c29fa93f8458e7a3b2f8996bbd13ff023f92e4a74f1d086d22af5e546b8\\"}\\n\\n\\ndef read_split(path, split):\\n    if split not in EXPECTED or Path(path).name==\'test.jsonl\':\\n        raise ValueError(\'Reserved test input is forbidden\')\\n    raw=Path(path).read_bytes()\\n    if hashlib.sha256(raw).hexdigest()!=EXPECTED[split]:raise ValueError(\'Frozen \'+split+\' hash mismatch\')\\n    rows=[json.loads(line) for line in raw.decode().splitlines() if line.strip()]\\n    if any(row[\'split\']!=split for row in rows):raise ValueError(\'Mixed split input\')\\n    return rows\\n\\n\\ndef write_json(path,value):\\n    path.write_text(json.dumps(value,ensure_ascii=False,indent=2)+\'\\\\n\',encoding=\'utf-8\')\\n\\n\\ndef write_lines(path,rows):\\n    path.write_text(\'\'.join(json.dumps(row,ensure_ascii=False)+\'\\\\n\' for row in rows),encoding=\'utf-8\')\\n\\n\\ndef synthetic():\\n    # Original synthetic examples; no paper text or model outputs.\\n    inputs=[(\'We use data to measure output. Firms report revenue.\',\'We use data to measure output; Firms report revenue.\'),\\n            (\'If demand falls, firms reduce output.\',\'Firms reduce output if demand falls.\'),\\n            (\'Some firms may not increase investment. We find no effect.\',\'Some firms may not increase investment. We find no effect.\')]\\n    train=[{\'id\':f\'synthetic-train-{i}\',\'paper_key\':\'SYN\',\'split\':\'train\',\'plain_english\':a,\'completion\':b,\'prompt\':a}\\n           for i,(a,b) in enumerate(inputs)]\\n    validation=[{\'id\':\'synthetic-validation\',\'paper_key\':\'SYN-HELD\',\'split\':\'validation\',\\n                 \'plain_english\':\'Firms report sales. Investors use the reports.\',\\n                 \'completion\':\'Firms report sales; Investors use the reports.\'}]\\n    return train,validation\\n\\n\\ndef execute(train, validation, directory, synthetic_run=False):\\n    if any(row[\'split\']!=\'train\' for row in train) or any(row[\'split\']!=\'validation\' for row in validation):\\n        raise ValueError(\'Split leakage detected\')\\n    if set(row[\'id\'] for row in train)&set(row[\'id\'] for row in validation):raise ValueError(\'Overlapping IDs\')\\n    if directory.exists() and any(directory.iterdir()):raise ValueError(\'Refusing to overwrite research outputs\')\\n    directory.mkdir(parents=True,exist_ok=True)\\n    profile=fit_profile(train)\\n    pair_diagnostics,phase_a=diagnose(train,profile)\\n    write_json(directory/\'A_train_signal_summary.json\',phase_a)\\n    write_json(directory/\'A_style_profile.json\',profile)\\n    write_lines(directory/\'A_pair_diagnostics.private.jsonl\',pair_diagnostics)\\n    outputs=[]\\n    for row in validation:\\n        result=rewrite(row[\'plain_english\'],profile)\\n        outputs.append({\'id\':row[\'id\'],**result,\\n                        \'reference_risk_screen\':risk_flags(row[\'plain_english\'],row[\'completion\'])})\\n    write_lines(directory/\'B_validation_prototype.private.jsonl\',outputs)\\n    phase_b={\'validation_rows\':len(validation),\'generated_candidates\':sum(len(r[\'candidates\']) for r in outputs),\\n             \'surface_accepted_candidates\':sum(c[\'verification\'][\'status\']==\'surface_preserved\' for r in outputs for c in r[\'candidates\']),\\n             \'selected_rewrites\':sum(r[\'status\']==\'selected_bounded_rewrite\' for r in outputs),\\n             \'abstained_unchanged\':sum(r[\'status\']==\'abstained_unchanged\' for r in outputs),\\n             \'mean_selected_source_overlap\':sum(overlap(row[\'plain_english\'],r[\'selected_text\']) for row,r in zip(validation,outputs))/len(outputs),\\n             \'mean_selected_style_gain_proxy\':sum(distance(row[\'plain_english\'],profile)-distance(r[\'selected_text\'],profile)\\n                                                   for row,r in zip(validation,outputs))/len(outputs),\\n             \'model_generation_executed\':False,\'human_judgment_used\':False,\\n             \'semantic_success_rate\':None,\'semantic_success_reason\':\'No calibrated semantic parser/NLI; bounded surface checks do not establish general entailment\',\\n             \'test_rows_used\':0}\\n    write_json(directory/\'B_prototype_summary.json\',phase_b)\\n    training=prepare(train,profile)\\n    for name,key in ((\'C_A_sft_candidates.private.jsonl\',\'sft\'),(\'C_B_weak_preferences.private.jsonl\',\'preferences\'),\\n                     (\'C_B_quarantined_preferences.private.jsonl\',\'quarantined_preferences\'),(\'C_parent_coverage.private.jsonl\',\'coverage\')):\\n        write_lines(directory/name,training[key])\\n    write_json(directory/\'C_preparation_summary.json\',training[\'summary\'])\\n    write_json(directory/\'C_experiment_plans.json\',plans())\\n    summary={\'run_kind\':\'synthetic_dry_run\' if synthetic_run else \'executed_cpu_research\',\\n             \'phase_a\':phase_a,\'phase_b\':phase_b,\'phase_c\':training[\'summary\'],\\n             \'gpu_jobs_submitted\':0,\'gpu_training_executed\':False,\'test_rows_used\':0,\\n             \'existing_experiments_modified\':False}\\n    write_json(directory/\'summary.json\',summary)\\n    files={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in directory.iterdir() if p.is_file()}\\n    write_json(directory/\'manifest.json\',{\'implementation\':\'claim_style_extension_v1\',\\n        \'inputs\':{\'training_rows\':len(train),\'validation_rows\':len(validation),\'training_split_sha256\':None if synthetic_run else EXPECTED[\'train\'],\\n                  \'validation_split_sha256\':None if synthetic_run else EXPECTED[\'validation\']},\\n        \'code_hashes\':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(__file__).parent.glob(\'*.py\')},\\n        \'files\':files,\'no_model_or_api_calls\':True,\'test_rows_used\':0})\\n    print(json.dumps({\'run_kind\':summary[\'run_kind\'],\'train_rows\':len(train),\'validation_rows\':len(validation),\\n                      \'selected_prototype_rewrites\':phase_b[\'selected_rewrites\'],\\n                      \'sft_candidates\':training[\'summary\'][\'sft_variants_staged\'],\\n                      \'training_ready\':False,\'gpu_jobs_submitted\':0,\'output_dir\':str(directory)}))\\n    return summary\\n\\n\\ndef main(argv=None):\\n    parser=argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\'--dry-run\',action=\'store_true\',help=\'Use only synthetic examples; CPU, no model/API\')\\n    parser.add_argument(\'--output-dir\',type=Path,required=True)\\n    args=parser.parse_args(argv)\\n    train,validation=synthetic() if args.dry_run else (read_split(ROOT/\'data/fama_corpus_v1/train.jsonl\',\'train\'),\\n                                                   read_split(ROOT/\'data/fama_corpus_v1/validation.jsonl\',\'validation\'))\\n    execute(train,validation,args.output_dir,args.dry_run)\\n\\n\\nif __name__==\'__main__\':main()\\n", "fama_style/research/signal.py": "\\"\\"\\"Training-only feature profiles and distributions, without ML dependencies.\\"\\"\\"\\nimport collections\\nimport difflib\\nimport math\\nimport re\\nimport statistics\\n\\nfrom .claims import sentences, tokens, risk_flags\\n\\nFEATURES = (\\"words_per_sentence\\", \\"semicolon_rate\\", \\"author_voice_rate\\", \\"initial_if_rate\\", \\"parenthesis_rate\\")\\n\\n\\ndef features(text):\\n    words = tokens(text)\\n    parts = sentences(text)\\n    n = max(1, len(words))\\n    return {\\"words_per_sentence\\": len(words)/max(1, len(parts)),\\n            \\"semicolon_rate\\": 100*text.count(\';\')/n,\\n            \\"author_voice_rate\\": 100*len(re.findall(r\\"\\\\b(?:we|our)\\\\b\\", text, re.I))/n,\\n            \\"initial_if_rate\\": sum(p.lower().startswith(\'if \') for p in parts)/max(1,len(parts)),\\n            \\"parenthesis_rate\\": 100*text.count(\'(\')/n}\\n\\n\\ndef overlap(a, b):\\n    return difflib.SequenceMatcher(None, tokens(a), tokens(b), autojunk=False).ratio()\\n\\n\\ndef distribution(values, bins=None):\\n    ordered = sorted(values)\\n    def quantile(q):\\n        position = (len(ordered)-1)*q\\n        lo, hi = math.floor(position), math.ceil(position)\\n        return ordered[lo]+(ordered[hi]-ordered[lo])*(position-lo)\\n    report = {\\"n\\": len(values), \\"min\\": ordered[0], \\"p10\\": quantile(.1), \\"p25\\": quantile(.25),\\n              \\"median\\": quantile(.5), \\"p75\\": quantile(.75), \\"p90\\": quantile(.9), \\"max\\": ordered[-1],\\n              \\"mean\\": statistics.mean(values)}\\n    if bins:\\n        report[\\"histogram\\"] = [{\\"lower\\": a, \\"upper\\": b, \\"count\\": sum(a<=v<b or (i==len(bins)-2 and v==b) for v in values)}\\n                               for i,(a,b) in enumerate(zip(bins,bins[1:]))]\\n    return report\\n\\n\\ndef fit_profile(rows):\\n    if not rows or any(row[\'split\']!=\'train\' for row in rows):\\n        raise ValueError(\'Style features can only be fitted on training references\')\\n    source = [features(row[\'plain_english\']) for row in rows]\\n    target = [features(row[\'completion\']) for row in rows]\\n    profile = {\\"fit_split\\": \\"train\\", \\"training_rows\\": len(rows), \\"features\\": {},\\n               \\"limitations\\": \\"Surface style proxies, not an author classifier; topic/length confounds remain\\"}\\n    for key in FEATURES:\\n        a,b=[r[key] for r in source],[r[key] for r in target]\\n        profile[\'features\'][key] = {\\"source_mean\\": statistics.mean(a), \\"reference_mean\\": statistics.mean(b),\\n                                    \\"reference_std\\": max(statistics.pstdev(b), .1),\\n                                    \\"mean_paired_delta\\": statistics.mean(y-x for x,y in zip(a,b)),\\n                                    \\"positive_pair_fraction\\": sum(y>x for x,y in zip(a,b))/len(a),\\n                                    \\"negative_pair_fraction\\": sum(y<x for x,y in zip(a,b))/len(a)}\\n    # Rules are eligible only when their direction follows the observed training delta.\\n    f=profile[\'features\'];ops=[]\\n    if f[\'words_per_sentence\'][\'mean_paired_delta\']>.5:ops.append(\'join_independent_sentences\')\\n    if f[\'words_per_sentence\'][\'mean_paired_delta\']<-.5:ops.append(\'split_independent_clauses\')\\n    if f[\'initial_if_rate\'][\'mean_paired_delta\']>.01:ops.append(\'condition_to_front\')\\n    if f[\'initial_if_rate\'][\'mean_paired_delta\']<-.01:ops.append(\'condition_to_end\')\\n    if f[\'author_voice_rate\'][\'mean_paired_delta\']<-.05:ops.append(\'neutralize_author_voice\')\\n    profile[\'eligible_operations\']=ops\\n    return profile\\n\\n\\ndef distance(text, profile):\\n    observed=features(text)\\n    return statistics.mean(abs(observed[key]-value[\'reference_mean\'])/value[\'reference_std\']\\n                           for key,value in profile[\'features\'].items())\\n\\n\\ndef diagnose(rows, profile):\\n    result=[]\\n    for row in rows:\\n        a,b=row[\'plain_english\'],row[\'completion\'];fa,fb=features(a),features(b)\\n        result.append({\\"id\\":row[\'id\'],\\"paper_key\\":row[\'paper_key\'],\\"lexical_overlap\\":overlap(a,b),\\n                       \\"word_length_ratio\\":len(tokens(b))/max(1,len(tokens(a))),\\n                       \\"feature_deltas\\":{key:fb[key]-fa[key] for key in FEATURES},\\n                       \\"style_vector_magnitude\\":statistics.mean(abs(fb[key]-fa[key])/profile[\'features\'][key][\'reference_std\'] for key in FEATURES),\\n                       \\"alignment_risk_flags\\":risk_flags(a,b),\\n                       \\"alignment_status\\":\\"risk_screen_only_not_entailment\\"})\\n    summary={\\"rows\\":len(rows),\\"lexical_overlap\\":distribution([r[\'lexical_overlap\'] for r in result],[0,.2,.4,.6,.8,.85,1]),\\n             \\"word_length_ratio\\":distribution([r[\'word_length_ratio\'] for r in result]),\\n             \\"style_vector_magnitude\\":distribution([r[\'style_vector_magnitude\'] for r in result]),\\n             \\"feature_delta_distributions\\":{key:distribution([r[\'feature_deltas\'][key] for r in result]) for key in FEATURES},\\n             \\"alignment_risk_counts\\":dict(collections.Counter(flag for r in result for flag in r[\'alignment_risk_flags\'])),\\n             \\"pairs_with_surface_alignment_flags\\":sum(bool(r[\'alignment_risk_flags\']) for r in result),\\n             \\"exact_normalized_copies\\":sum(tokens(row[\'plain_english\'])==tokens(row[\'completion\']) for row in rows),\\n             \\"overlap_at_least_0_85\\":sum(r[\'lexical_overlap\']>=.85 for r in result),\\n             \\"per_paper\\":{key:{\\"rows\\":len(group),\\"median_lexical_overlap\\":statistics.median(r[\'lexical_overlap\'] for r in group),\\n                               \\"median_style_magnitude\\":statistics.median(r[\'style_vector_magnitude\'] for r in group),\\n                               \\"mean_feature_deltas\\":{feature:statistics.mean(r[\'feature_deltas\'][feature] for r in group) for feature in FEATURES}}\\n                          for key in sorted({r[\'paper_key\'] for r in result})\\n                          for group in [[r for r in result if r[\'paper_key\']==key]]},\\n             \\"semantic_alignment_caveat\\":\\"Counts/lexical features flag risk; differing surface markers can be equivalent and matching markers can hide drift.\\"}\\n    return result, summary\\n", "fama_style/rhetoric/__init__.py": "\\"\\"\\"Separate, CPU-first rhetorical structure and oracle inference experiment.\\"\\"\\"\\n", "fama_style/rhetoric/experiment.py": "\\"\\"\\"Prepare rhetorical pairs and run an explicitly requested inference-only oracle.\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport re\\nimport threading\\nimport time\\n\\nfrom fama_style.research.run_extension import read_split, synthetic, EXPECTED\\nfrom fama_style.research.signal import fit_profile\\nfrom fama_style.research.claims import sha\\nfrom .extract import paired, inspect_legacy\\nfrom .mapping import probe\\nfrom .oracle import jobs_for, generate_jobs, validate_outputs, score, PROTOCOL\\nfrom .schema import VERSION\\nfrom .semantics import NLI_MODEL, assess, calibration_controls\\n\\nROOT = Path(__file__).resolve().parents[3]\\nMODEL_ID = \'google/gemma-3-4b-it\'\\nMODEL_REVISION = \'093f9f388b31de276ce2de164bdc2081324b9767\'\\nMAX_NEW_TOKENS = 512\\nCONTEXT_LIMIT = 4096\\n\\n\\ndef write_json(path, value):\\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2)+\'\\\\n\', encoding=\'utf-8\')\\n    temporary.replace(path)\\n\\n\\ndef write_lines(path, rows):\\n    path.write_text(\'\'.join(json.dumps(row,ensure_ascii=False)+\'\\\\n\' for row in rows), encoding=\'utf-8\')\\n\\n\\ndef fresh(path):\\n    if path.exists() and any(path.iterdir()): raise ValueError(\'Refusing to overwrite experiment outputs\')\\n    path.mkdir(parents=True, exist_ok=True)\\n\\n\\ndef manifest(directory, synthetic_run):\\n    # Hash only outputs/code; do not load any reserved-test records.\\n    paths = sorted(Path(__file__).resolve().parent.glob(\'*.py\'))\\n    paths += sorted((ROOT/\'src/fama_style/research\').glob(\'*.py\'))\\n    paths += [ROOT/\'src/fama_style/prompting/corpus.py\']\\n    write_json(directory/\'manifest.json\', {\'protocol\': PROTOCOL, \'schema\': VERSION,\\n        \'synthetic\': synthetic_run, \'test_rows_used\': 0,\\n        \'input_split_hashes\': None if synthetic_run else EXPECTED,\\n        \'code_hashes\': {str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in paths},\\n        \'files\': {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(directory.iterdir())\\n                  if p.is_file() and p.name!=\'manifest.json\'}})\\n\\n\\ndef prepare(train, validation, output_dir, dry_run=False):\\n    if any(row[\'split\']!=\'train\' for row in train) or any(row[\'split\']!=\'validation\' for row in validation):\\n        raise ValueError(\'Unexpected split in rhetorical experiment\')\\n    if len({r[\'id\'] for r in train+validation})!=len(train)+len(validation): raise ValueError(\'Duplicate or overlapping split IDs\')\\n    fresh(output_dir)\\n    pairs = [{\'id\': row[\'id\'], \'paper_key\': row[\'paper_key\'], \'split\':\'train\',\\n              \'annotation\': paired(row[\'plain_english\'], row[\'completion\'])} for row in train]\\n    write_lines(output_dir/\'paired_schemas.private.jsonl\', pairs)\\n    analysis = probe(pairs)\\n    write_json(output_dir/\'learnability_summary.json\', analysis)\\n    write_json(output_dir/\'legacy_inventory.json\', inspect_legacy(train))\\n    jobs = jobs_for(validation)\\n    write_lines(output_dir/\'oracle_jobs.private.jsonl\', jobs)\\n    protocol = {\'protocol\': PROTOCOL, \'status\':\'prepared_not_executed\' if not dry_run else \'synthetic_wiring_check\',\\n        \'training_pairs\': len(train), \'validation_examples\': len(validation), \'generation_jobs\': len(jobs),\\n        \'schema_version\': VERSION, \'model_id\': MODEL_ID, \'model_revision\': MODEL_REVISION,\\n        \'decoding\': {\'do_sample\': False, \'max_new_tokens\':MAX_NEW_TOKENS, \'use_cache\':True, \'seed\':42},\\n        \'context_limit\':CONTEXT_LIMIT, \'truncate\':False, \'condition_order\':\'cyclic A/B/C per validation index\',\\n        \'claim_information\':\'B/C share source evidence units; A is the ordinary frozen prompt\',\\n        \'primary_comparison\':\'C versus B; A anchors the ordinary rewrite baseline\',\\n        \'oracle_reference_wording_in_generator\':False, \'test_rows_used\':0, \'human_review_required\':False,\\n        \'gpu_training_allowed\':False, \'inference_executed\':False,\\n        \'semantic_assessment\': \'Independent bidirectional document and sentence-unit NLI after Gemma is unloaded; synthetic controls must pass; over-context pairs abstain\',\\n        \'factorial_followup\':{\'status\':\'blocked_until_oracle_support\', \'cells\':[\\n            {\'weights\':\'base\',\'mapping\':\'none\'}, {\'weights\':\'base\',\'mapping\':\'learned\'},\\n            {\'weights\':\'frozen_qlora\',\'mapping\':\'none\'}, {\'weights\':\'frozen_qlora\',\'mapping\':\'learned\'}],\\n            \'controls\':\'same source examples, model revision, quantization, budgets, prompts within mapping condition; existing adapter was not plan-conditioned\'}}\\n    write_json(output_dir/\'oracle_protocol.json\', protocol)\\n    profile = fit_profile(train)\\n    write_json(output_dir/\'style_profile.json\', profile)\\n    if dry_run:\\n        sources = {row[\'id\']: row[\'plain_english\'] for row in validation}\\n        outputs = generate_jobs(jobs, lambda job: {\'output\':sources[job[\'id\']], \'hit_output_limit\':False, \'synthetic\':True,\\n                                                    \'input_tokens\':None, \'output_tokens\':None, \'generation_seconds\':None})\\n        records, summary = score(validation, jobs, outputs, profile, synthetic=True)\\n        write_lines(output_dir/\'synthetic_outputs.private.jsonl\', outputs)\\n        write_json(output_dir/\'synthetic_wiring_summary.json\', summary)\\n    manifest(output_dir, dry_run)\\n    return jobs, profile, analysis\\n\\n\\ndef secret_token():\\n    if Path(\'/kaggle\').exists():\\n        try:\\n            from kaggle_secrets import UserSecretsClient\\n            value = UserSecretsClient().get_secret(\'HF_TOKEN\')\\n        except Exception:\\n            raise RuntimeError(\'Enable the Kaggle Secret named HF_TOKEN\') from None\\n    else:\\n        value = os.environ.get(\'HF_TOKEN\')\\n    if not isinstance(value,str) or not value.strip(): raise RuntimeError(\'Configure HF_TOKEN outside code and repository\')\\n    return value\\n\\n\\ndef with_heartbeat(call, label):\\n    stop = threading.Event(); started = time.monotonic()\\n    def tick():\\n        while not stop.wait(30): print(f\'{label}: working ({int(time.monotonic()-started)} seconds)\', flush=True)\\n    thread = threading.Thread(target=tick, daemon=True); thread.start()\\n    try: return call()\\n    finally: stop.set(); thread.join(timeout=1)\\n\\n\\ndef audit_input_ids(job, ids):\\n    if sha(job[\'prompt\'])!=job[\'prompt_sha256\']: raise ValueError(\'Changed prompt\')\\n    if not ids or len(ids)+MAX_NEW_TOKENS>CONTEXT_LIMIT: raise ValueError(\'Prompt/context overflow; no truncation allowed\')\\n    return {\'id\':job[\'id\'],\'condition\':job[\'condition\'],\'input_tokens\':len(ids),\\n            \'prompt_sha256\':job[\'prompt_sha256\'],\'input_ids_sha256\':sha(json.dumps(ids))}\\n\\n\\ndef run_training(validation, jobs, profile, output_dir):\\n    \\"\\"\\"Inference only; name follows the repository convention for lazy ML imports.\\"\\"\\"\\n    fresh(output_dir)\\n    status = {\'complete\':False, \'training_called\':False, \'test_rows_used\':0, \'protocol\':PROTOCOL}\\n    started = time.monotonic()\\n    def phase(name, **extra):\\n        status.update(phase=name, **extra); write_json(output_dir/\'run_status.json\',status)\\n        print(\'Phase: \'+name, flush=True)\\n    try:\\n        phase(\'loading_inference_runtime\')\\n        os.environ[\'CUDA_VISIBLE_DEVICES\']=\'0\'\\n        os.environ[\'HF_HUB_DISABLE_PROGRESS_BARS\']=\'1\'\\n        os.environ[\'HF_HUB_DISABLE_TELEMETRY\']=\'1\'\\n        os.environ[\'TOKENIZERS_PARALLELISM\']=\'false\'\\n        import torch\\n        from transformers import (AutoProcessor, AutoTokenizer, AutoModelForSequenceClassification,\\n            BitsAndBytesConfig, Gemma3ForConditionalGeneration, LogitsProcessor, LogitsProcessorList)\\n        from huggingface_hub import HfApi\\n        from transformers.utils import logging\\n        logging.set_verbosity_error()\\n        if not torch.cuda.is_available() or torch.cuda.device_count()!=1:\\n            raise RuntimeError(\'Use Kaggle T4 x2; expose only its first GPU before importing torch\')\\n        torch.cuda.init(); torch.cuda.set_device(0); torch.manual_seed(42); torch.cuda.reset_peak_memory_stats(0)\\n        token = secret_token()\\n        phase(\'loading_pinned_processor\')\\n        processor = with_heartbeat(lambda:AutoProcessor.from_pretrained(MODEL_ID,revision=MODEL_REVISION,token=token), \'Processor load\')\\n        def inputs_for(job):\\n            if sha(job[\'prompt\'])!=job[\'prompt_sha256\']: raise ValueError(\'Changed prompt\')\\n            return processor.apply_chat_template([{\'role\':\'user\',\'content\':[{\'type\':\'text\',\'text\':job[\'prompt\']}]}],\\n                tokenize=True,add_generation_prompt=True,truncation=False,return_dict=True,return_tensors=\'pt\')\\n        audits=[]\\n        for job in jobs:\\n            batch=inputs_for(job)\\n            audits.append(audit_input_ids(job,batch[\'input_ids\'][0].tolist()))\\n        write_json(output_dir/\'token_audit.json\',audits)\\n        phase(\'loading_pinned_gemma\')\\n        model=with_heartbeat(lambda:Gemma3ForConditionalGeneration.from_pretrained(MODEL_ID,revision=MODEL_REVISION,\\n            token=token,dtype=torch.float32,device_map={\'\':0},attn_implementation=\'sdpa\',\\n            quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type=\'nf4\',\\n                bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float32)), \'Gemma load\')\\n        del token\\n        if any(t.device.type!=\'cuda\' or t.device.index!=0 for t in list(model.parameters())+list(model.buffers())):\\n            raise RuntimeError(\'Unexpected model tensor placement\')\\n        model.eval(); model.requires_grad_(False)\\n        class FiniteScores(LogitsProcessor):\\n            def __call__(self,input_ids,scores):\\n                if bool(torch.isnan(scores).any() | torch.isposinf(scores).any() | ~torch.isfinite(scores).any(dim=-1).all()):\\n                    raise RuntimeError(\'Nonfinite generation scores\')\\n                return scores\\n        audit_by_key={(a[\'id\'],a[\'condition\']):a for a in audits}\\n        def generate(job):\\n            batch=inputs_for(job); n=int(batch[\'input_ids\'].shape[-1]); audit=audit_by_key[(job[\'id\'],job[\'condition\'])]\\n            if sha(json.dumps(batch[\'input_ids\'][0].tolist()))!=audit[\'input_ids_sha256\']: raise ValueError(\'Prompt tokenization changed\')\\n            batch={k:v.to(model.get_input_embeddings().weight.device) for k,v in batch.items()}\\n            phase(\'generation\',completed_jobs=generate.completed,total_jobs=len(jobs))\\n            before=time.monotonic()\\n            with torch.inference_mode():\\n                out=with_heartbeat(lambda:model.generate(**batch,do_sample=False,max_new_tokens=MAX_NEW_TOKENS,\\n                    use_cache=True,pad_token_id=processor.tokenizer.pad_token_id,\\n                    logits_processor=LogitsProcessorList([FiniteScores()]),remove_invalid_values=False), \'Generation\')\\n            new=out[0,n:]; eos=model.generation_config.eos_token_id; eos=eos if isinstance(eos,list) else [eos]\\n            result={\'output\':processor.tokenizer.decode(new,skip_special_tokens=True).strip(),\\n                    \'input_tokens\':n,\'output_tokens\':int(new.numel()),\'generation_seconds\':time.monotonic()-before,\\n                    \'hit_output_limit\':bool(new.numel()>=MAX_NEW_TOKENS and int(new[-1]) not in eos),\'synthetic\':False}\\n            generate.completed+=1\\n            return result\\n        generate.completed=0\\n        outputs=generate_jobs(jobs,generate,lambda rows:write_lines(output_dir/\'generations.private.jsonl\',rows))\\n        validate_outputs(jobs,outputs)\\n        gpu_memory={\'peak_allocated_bytes\':torch.cuda.max_memory_allocated(0),\'peak_reserved_bytes\':torch.cuda.max_memory_reserved(0)}\\n        del generate,model,processor,batch\\n        torch.cuda.empty_cache()\\n        phase(\'loading_independent_nli_on_cpu\')\\n        nli_started=time.monotonic()\\n        revision=HfApi(token=False).model_info(NLI_MODEL,timeout=30).sha\\n        if not isinstance(revision,str) or not re.fullmatch(\'[0-9a-f]{40}\',revision): raise ValueError(\'Unresolved NLI revision\')\\n        tokenizer=with_heartbeat(lambda:AutoTokenizer.from_pretrained(NLI_MODEL,revision=revision,token=False), \'NLI tokenizer load\')\\n        nli=with_heartbeat(lambda:AutoModelForSequenceClassification.from_pretrained(NLI_MODEL,revision=revision,token=False), \'NLI model load\')\\n        nli.eval(); nli.requires_grad_(False); torch.set_num_threads(min(4,os.cpu_count() or 1))\\n        labels={int(k):v.lower() for k,v in nli.config.id2label.items()}\\n        if set(labels.values())!={\'entailment\',\'neutral\',\'contradiction\'}: raise ValueError(\'Unexpected NLI labels\')\\n        limit=min(512,int(getattr(nli.config,\'max_position_embeddings\',512)),int(tokenizer.model_max_length))\\n        def predict(premise,hypothesis):\\n            batch=tokenizer(premise,hypothesis,truncation=False,return_tensors=\'pt\')\\n            if int(batch[\'input_ids\'].shape[-1])>limit:return None\\n            with torch.inference_mode(): probabilities=torch.softmax(nli(**batch).logits[0],dim=-1).tolist()\\n            return {labels[i]:float(p) for i,p in enumerate(probabilities)}\\n        calibration=with_heartbeat(lambda:calibration_controls(predict),\'NLI synthetic controls\')\\n        write_json(output_dir/\'nli_controls.json\',calibration)\\n        sources={row[\'id\']:row[\'plain_english\'] for row in validation}; semantic={}\\n        for index,row in enumerate(outputs,1):\\n            semantic[(row[\'id\'],row[\'condition\'])]=with_heartbeat(lambda:assess(sources[row[\'id\']],row[\'output\'],predict),\'NLI assessment\')\\n            phase(\'independent_semantic_assessment\',completed_nli_outputs=index,total_jobs=len(outputs))\\n        records,summary=score(validation,jobs,outputs,profile,semantic=semantic,calibration=calibration)\\n        write_lines(output_dir/\'metrics.private.jsonl\',records)\\n        summary.update(model_id=MODEL_ID,model_revision=MODEL_REVISION,nli_model=NLI_MODEL,nli_revision=revision,\\n            nli_seconds=time.monotonic()-nli_started,total_seconds=time.monotonic()-started,memory=gpu_memory,\\n            versions={name:importlib.metadata.version(name) for name in (\'torch\',\'transformers\',\'bitsandbytes\',\'accelerate\',\'huggingface-hub\')})\\n        write_json(output_dir/\'evaluation.json\',summary)\\n        phase(\'complete\',complete=True)\\n        manifest(output_dir,False)\\n    except Exception as exc:\\n        # Remote exception text/locals can contain authenticated URLs. Do not persist them.\\n        frames=[]; tb=exc.__traceback__\\n        while tb:\\n            if tb.tb_frame.f_code.co_filename==__file__:frames.append(tb.tb_lineno)\\n            tb=tb.tb_next\\n        phase(\'failed\',complete=False,error_code=\'oracle_inference_failed\',local_line_numbers=frames)\\n        raise RuntimeError(\'Oracle inference failed; inspect run_status.json and saved partial outputs. No training was called.\') from None\\n\\n\\ndef main(argv=None):\\n    parser=argparse.ArgumentParser(description=__doc__)\\n    action=parser.add_mutually_exclusive_group()\\n    action.add_argument(\'--dry-run\',action=\'store_true\',help=\'Synthetic CPU wiring check, no model/Secret/network\')\\n    action.add_argument(\'--execute-inference\',action=\'store_true\',help=\'Explicitly run the 63-job Kaggle inference experiment; never training\')\\n    parser.add_argument(\'--train\',type=Path,default=ROOT/\'data/fama_corpus_v1/train.jsonl\')\\n    parser.add_argument(\'--validation\',type=Path,default=ROOT/\'data/fama_corpus_v1/validation.jsonl\')\\n    parser.add_argument(\'--output-dir\',required=True,type=Path)\\n    args=parser.parse_args(argv)\\n    if args.dry_run:\\n        train,validation=synthetic()\\n        for index,row in enumerate(train):row[\'paper_key\']=f\'SYN-{index}\'\\n    else:train,validation=read_split(args.train,\'train\'),read_split(args.validation,\'validation\')\\n    fresh(args.output_dir)\\n    jobs,profile,analysis=prepare(train,validation,args.output_dir/\'preparation\',args.dry_run)\\n    if args.execute_inference:run_training(validation,jobs,profile,args.output_dir/\'inference\')\\n    print(json.dumps({\'status\':\'synthetic_check_complete\' if args.dry_run else \'inference_complete\' if args.execute_inference else \'prepared_not_executed\',\\n                      \'training_pairs\':len(train),\'validation_examples\':len(validation),\'jobs\':len(jobs),\'training_called\':False,\'test_rows_used\':0}))\\n\\n\\nif __name__==\'__main__\':main()\\n", "fama_style/rhetoric/extract.py": "\\"\\"\\"Independent surface annotation, paired hypotheses, and constrained operations.\\"\\"\\"\\nimport collections\\nimport json\\nimport re\\n\\nfrom fama_style.research.claims import ledger, sha, tokens, sentences, conditional, variants, verify, risk_flags\\nfrom fama_style.research.signal import overlap\\nfrom .schema import VERSION, EXTRACTOR, validate_links, validate\\n\\nPROTECTED_FEATURES = {\'hedging\', \'citation\'}\\n\\n\\ndef count_bin(n):\\n    return \'one\' if n == 1 else \'two\' if n == 2 else \'three_plus\'\\n\\n\\ndef rate_bin(n, total):\\n    return \'none\' if not n else \'low\' if n/max(1, total) <= .025 else \'high\'\\n\\n\\ndef role(text):\\n    \\"\\"\\"Cue observations, never assertions that a relation is semantically valid.\\"\\"\\"\\n    low = text.lower()\\n    if re.search(r\'\\\\b(?:is defined as|we define|denotes)\\\\b\', low): return \'definition\'\\n    if re.match(r\'(?:however|nevertheless|in contrast|but)\\\\b\', low): return \'contrast\'\\n    if re.match(r\'(?:therefore|thus|hence|in sum)\\\\b\', low): return \'conclusion\'\\n    if low.startswith((\'if \', \'unless \', \'when \')): return \'condition\'\\n    if re.search(r\'\\\\b(?:we find|results show|table \\\\d|we estimate|we test)\\\\b\', low): return \'reporting\'\\n    return \'assertion\'\\n\\n\\ndef extract(text):\\n    if not isinstance(text, str) or not text.strip(): raise ValueError(\'Empty paragraph\')\\n    claims = ledger(text)\\n    parts = sentences(text)\\n    units, moves = [], []\\n    for index, part in enumerate(parts, 1):\\n        subordinated = bool(re.search(r\'\\\\b(if|unless|when|although|because|which|that|while|whereas)\\\\b\', part, re.I))\\n        coordinated = bool(re.search(r\';|,\\\\s+(?:and|but|or)\\\\b\', part, re.I))\\n        syntax = \'mixed\' if subordinated and coordinated else \'subordinated\' if subordinated else \'coordinated\' if coordinated else \'simple\'\\n        position = \'none\'\\n        if re.search(r\'\\\\b(if|unless|when|provided that)\\\\b\', part, re.I):\\n            position = \'front\' if re.match(r\'(if|unless|when|provided that)\\\\b\', part, re.I) else \'end\' if conditional(part) else \'embedded\'\\n        n = len(tokens(part)); unit = f\'s{index}\'; refs = [f\'c{index}\']\\n        units.append({\'unit\': unit, \'claims\': refs, \'syntax\': syntax,\\n                      \'condition_position\': position, \'voice\': \'present\' if re.search(r\'\\\\b(we|our|us)\\\\b\', part, re.I) else \'absent\',\\n                      \'length\': \'short\' if n <= 16 else \'medium\' if n <= 32 else \'long\'})\\n        moves.append({\'unit\': unit, \'claims\': refs, \'role\': role(part)})\\n    roles = {move[\'role\'] for move in moves}\\n    specials = roles & {\'contrast\', \'condition\', \'reporting\'}\\n    organization = \'mixed\' if len(specials)>1 else \'contrastive\' if \'contrast\' in roles else \'conditional\' if \'condition\' in roles else \'reporting\' if \'reporting\' in roles else \'single\' if len(parts)==1 else \'sequence\'\\n    words = tokens(text); n = len(words)\\n    structure = {\'version\': VERSION,\\n        \'paragraph\': {\'organization\': organization, \'sentence_count\': count_bin(len(parts)), \'moves\': moves},\\n        \'sentences\': units,\\n        \'language\': {\'author_voice\': \'present\' if any(unit[\'voice\']==\'present\' for unit in units) else \'absent\',\\n            \'semicolon\': rate_bin(text.count(\';\'), n), \'parenthesis\': rate_bin(text.count(\'(\'), n),\\n            \'hedging\': rate_bin(len(re.findall(r\'\\\\b(may|might|could|possibly|perhaps)\\\\b\', text, re.I)), n),\\n            \'citation\': rate_bin(len(re.findall(r\'\\\\b(?:18|19|20)\\\\d{2}\\\\b\', text)), n),\\n            \'nominalization\': rate_bin(sum(bool(re.search(r\'(tion|sion|ment|ance|ence|ity)$\', word)) for word in words), n)}}\\n    validate_links(structure, claims)\\n    return {\'structure\': structure, \'semantic_ledger\': claims,\\n        \'provenance\': {\'specification\': VERSION, \'extractor\': EXTRACTOR, \'input_sha256\': sha(text),\\n                       \'annotation_method\': \'deterministic_surface_rules\', \'independent_of_paired_text\': True},\\n        \'checks\': {\'claim_links_valid\': True, \'semantic_parse_verified\': False,\\n                   \'atomic_claim_coverage\': \'unresolved\', \'unsupported_relation_creation_allowed\': False}}\\n\\n\\ndef flatten(structure):\\n    validate(structure)\\n    sentence = structure[\'sentences\']\\n    def majority(key):\\n        counts = collections.Counter(row[key] for row in sentence)\\n        return sorted(counts, key=lambda item:(-counts[item], item))[0]\\n    return {\'organization\': structure[\'paragraph\'][\'organization\'], \'sentence_count\': structure[\'paragraph\'][\'sentence_count\'],\\n            \'syntax\': majority(\'syntax\'), \'condition_position\': majority(\'condition_position\'),\\n            \'sentence_length\': majority(\'length\'), **structure[\'language\']}\\n\\n\\ndef paired(source, target):\\n    \\"\\"\\"Extract separately; alignment is an explicitly unverified lexical hypothesis.\\"\\"\\"\\n    a, b = extract(source), extract(target)\\n    hypotheses = []\\n    for target_claim in b[\'semantic_ledger\'][\'claims\']:\\n        ranked = sorted(((overlap(claim[\'evidence\'][\'text\'], target_claim[\'evidence\'][\'text\']), claim[\'claim_id\'])\\n                         for claim in a[\'semantic_ledger\'][\'claims\']), reverse=True)\\n        score, claim_id = ranked[0]\\n        hypotheses.append({\'target_claim\': target_claim[\'claim_id\'], \'source_claim\': claim_id if score >= .35 else None,\\n                           \'similarity\': score, \'status\': \'lexical_hypothesis_not_semantically_validated\'})\\n    fa, fb = flatten(a[\'structure\']), flatten(b[\'structure\'])\\n    refs = [claim[\'claim_id\'] for claim in a[\'semantic_ledger\'][\'claims\']]\\n    changes = [{\'operation\': \'preserve_source_feature\' if key in PROTECTED_FEATURES else \'set_template_feature\',\\n                \'feature\': key, \'from\': fa[key], \'to\': fa[key] if key in PROTECTED_FEATURES else fb[key],\\n                \'observed_target\': fb[key], \'status\': \'blocked_semantic_sensitive\' if key in PROTECTED_FEATURES else \'structural_hypothesis\',\\n                \'source_claim_refs\': refs, \'scope\': \'planning_only\', \'introduces_semantic_relation\': False}\\n               for key in fa if fa[key] != fb[key]]\\n    # Programs are replayable surface controls, not a claim that they realize the target.\\n    programs = [{\'operation\': item[\'operation\'], \'source_claim_refs\': refs,\\n                 \'output_sha256\': sha(item[\'text\']), \'verification\': verify(source, item[\'text\']),\\n                 \'target_feature_matches\': sum(value == flatten(extract(item[\'text\'])[\'structure\'])[key] for key,value in fb.items())}\\n                for item in variants(source) if item[\'operation\']!=\'identity\']\\n    return {\'source\': a, \'target\': b, \'claim_alignment\': hypotheses, \'template_changes\': changes,\\n            \'executable_surface_controls\': programs,\\n            \'consistency\': {\'source_target_surface_flags\': risk_flags(source, target),\\n                            \'semantic_alignment_verified\': False, \'claim_reordering_allowed\': False}}\\n\\n\\ndef execute_operation(text, operation, claim_refs, expected_source_sha):\\n    \\"\\"\\"An operation can touch only this source\'s known claims and bounded grammar.\\"\\"\\"\\n    if sha(text) != expected_source_sha: raise ValueError(\'Changed operation source\')\\n    known = {c[\'claim_id\'] for c in ledger(text)[\'claims\']}\\n    if not claim_refs or set(claim_refs) != known: raise ValueError(\'Invalid operation claim scope\')\\n    candidate = next((item for item in variants(text) if item[\'operation\']==operation), None)\\n    if candidate is None or verify(text, candidate[\'text\'])[\'status\']!=\'surface_preserved\':\\n        raise ValueError(\'Unsupported semantic or structural operation\')\\n    return candidate[\'text\']\\n\\n\\ndef inspect_legacy(rows):\\n    shapes = collections.Counter(); labels = collections.Counter()\\n    for row in rows:\\n        value = row.get(\'rhetorical_taxonomy\')\\n        shape = \'missing\' if value is None else \',\'.join(sorted(value)) if isinstance(value, dict) else \'unexpected\'\\n        shapes[shape] += 1\\n        if isinstance(value, dict):\\n            # Inventory only pre-existing finite objectives, not private descriptions.\\n            top = value.get(\'level_1\', value.get(\'R1\', {}))\\n            objective = top.get(\'discourse_objective\') if isinstance(top, dict) else None\\n            if isinstance(objective, str) and re.fullmatch(r\'[A-Z_]{1,32}\', objective): labels[objective] += 1\\n    return {\'rows\': len(rows), \'formats\': dict(shapes), \'objective_counts\': dict(labels),\\n            \'reused_as_source_labels\': False, \'migration\': \'preserved unchanged; discourse objective/moves/relations do not match new three-level axes\'}\\n", "fama_style/rhetoric/mapping.py": "\\"\\"\\"Training-only learnability probes; no mapping is deployed before oracle evidence.\\"\\"\\"\\nimport collections\\nimport math\\nimport statistics\\n\\nfrom .extract import flatten, PROTECTED_FEATURES\\n\\n\\ndef entropy(values):\\n    counts = collections.Counter(values); n = sum(counts.values())\\n    return -sum(count/n * math.log2(count/n) for count in counts.values()) if n else 0.0\\n\\n\\ndef mode(values):\\n    counts = collections.Counter(values)\\n    return sorted(counts, key=lambda label: (-counts[label], label))[0]\\n\\n\\ndef probe(pairs):\\n    if not pairs or any(row[\'split\']!=\'train\' for row in pairs):\\n        raise ValueError(\'Only training pairs may inform mapping probes\')\\n    rows = [{\'paper\': row[\'paper_key\'], \'x\': flatten(row[\'annotation\'][\'source\'][\'structure\']),\\n             \'y\': flatten(row[\'annotation\'][\'target\'][\'structure\'])} for row in pairs]\\n    keys = [key for key in rows[0][\'x\'] if key not in PROTECTED_FEATURES]\\n    if len({row[\'paper\'] for row in rows}) < 2: raise ValueError(\'Need independent paper folds\')\\n    predictions = {name: [] for name in (\'source_identity\', \'fixed_prior\', \'retrieval\', \'smoothed_conditional\')}\\n    per_paper = {}\\n    for paper in sorted({row[\'paper\'] for row in rows}):\\n        fit = [row for row in rows if row[\'paper\']!=paper]\\n        held = [row for row in rows if row[\'paper\']==paper]\\n        local = {name: [] for name in predictions}\\n        for row in held:\\n            nearest = min(enumerate(fit), key=lambda item:(sum(row[\'x\'][key]!=item[1][\'x\'][key] for key in keys), item[0]))[1]\\n            pred = {name:{} for name in predictions}\\n            for key in keys:\\n                prior = collections.Counter(item[\'y\'][key] for item in fit)\\n                conditioned = collections.Counter(item[\'y\'][key] for item in fit if item[\'x\'][key]==row[\'x\'][key])\\n                # Five prior-equivalent observations regularize rare source buckets.\\n                labels = sorted(prior)\\n                conditional = max(labels, key=lambda label: conditioned[label] + 5*prior[label]/len(fit))\\n                pred[\'source_identity\'][key] = row[\'x\'][key]\\n                pred[\'fixed_prior\'][key] = mode(item[\'y\'][key] for item in fit)\\n                pred[\'retrieval\'][key] = nearest[\'y\'][key]\\n                pred[\'smoothed_conditional\'][key] = conditional\\n            for name in pred:\\n                matches = {key: pred[name][key]==row[\'y\'][key] for key in keys}\\n                predictions[name].append(matches); local[name].append(matches)\\n        per_paper[paper] = {name: statistics.mean(sum(record.values())/len(keys) for record in records)\\n                            for name, records in local.items()}\\n    scores = {name: {\'macro_field_accuracy\': statistics.mean(sum(row.values())/len(keys) for row in records),\\n                     \'per_field_accuracy\': {key: statistics.mean(row[key] for row in records) for key in keys}}\\n              for name, records in predictions.items()}\\n    return {\'rows\': len(rows), \'papers\': len(per_paper), \'fit_split\': \'train\',\\n        \'source_signatures\': len({tuple(row[\'x\'][key] for key in keys) for row in rows}),\\n        \'target_signatures\': len({tuple(row[\'y\'][key] for key in keys) for row in rows}),\\n        \'protected_features_excluded_from_prediction\': sorted(PROTECTED_FEATURES),\\n        \'per_field\': {key: {\'source_counts\': dict(collections.Counter(row[\'x\'][key] for row in rows)),\\n            \'target_counts\': dict(collections.Counter(row[\'y\'][key] for row in rows)),\\n            \'source_entropy_bits\': entropy([row[\'x\'][key] for row in rows]),\\n            \'target_entropy_bits\': entropy([row[\'y\'][key] for row in rows]),\\n            \'changed_fraction\': statistics.mean(row[\'x\'][key]!=row[\'y\'][key] for row in rows)} for key in keys},\\n        \'leave_one_training_paper_out\': scores, \'per_paper_macro_accuracy\': per_paper,\\n        \'protocol\': \'Exploratory categorical probes; validation and reserved test excluded; no deployable mapping fitted\',\\n        \'recommendation\': \'Test the oracle first. If supported, start with fixed prior; require paper-grouped gains before adding retrieval or smoothed conditional edits. No neural mapper yet.\',\\n        \'caveat\': \'Bucket prediction accuracy is not generation quality or semantic preservation; field imbalance and correlated controls inflate accuracy.\'}\\n", "fama_style/rhetoric/oracle.py": "\\"\\"\\"Content-free oracle templates, balanced prompts and automatic diagnostic scoring.\\"\\"\\"\\nimport copy\\nimport difflib\\nimport json\\nimport statistics\\n\\nfrom fama_style.prompting.corpus import make_prompt\\nfrom fama_style.research.claims import sha, verify, risk_flags, tokens\\nfrom fama_style.research.signal import overlap, distance\\nfrom .extract import extract, flatten\\nfrom .schema import validate, validate_links\\n\\nCONDITIONS = (\'A_plain\', \'B_source_structure\', \'C_oracle_target_structure\')\\nPROTOCOL = \'oracle_rhetoric_v1\'\\nPRESENTATION_KEYS = {\'organization\', \'sentence_count\', \'syntax\', \'condition_position\', \'sentence_length\',\\n                     \'author_voice\', \'semicolon\', \'parenthesis\', \'nominalization\'}\\n\\n\\ndef structural_template(structure):\\n    \\"\\"\\"Copy only validated finite labels; omit hashes, claims and reference spans.\\"\\"\\"\\n    validate(structure)\\n    return {\'paragraph\': {\'organization\': structure[\'paragraph\'][\'organization\'],\\n                          \'sentence_count\': structure[\'paragraph\'][\'sentence_count\'],\\n                          \'move_roles\': [item[\'role\'] for item in structure[\'paragraph\'][\'moves\']]},\\n            \'sentences\': [{key: item[key] for key in (\'syntax\', \'condition_position\', \'voice\', \'length\')}\\n                          for item in structure[\'sentences\']],\\n            \'language\': {key: value for key,value in structure[\'language\'].items() if key not in (\'hedging\',\'citation\')}}\\n\\n\\ndef adherence(actual, desired):\\n    \\"\\"\\"Score finite feature buckets and the requested paragraph/sentence sequences.\\"\\"\\"\\n    a, b = flatten(actual), flatten(desired)\\n    buckets = statistics.mean(a[key]==b[key] for key in PRESENTATION_KEYS)\\n    moves = difflib.SequenceMatcher(None, [r[\'role\'] for r in actual[\'paragraph\'][\'moves\']],\\n                                    [r[\'role\'] for r in desired[\'paragraph\'][\'moves\']], autojunk=False).ratio()\\n    def patterns(structure):\\n        return [tuple(row[key] for key in (\'syntax\',\'condition_position\',\'voice\',\'length\')) for row in structure[\'sentences\']]\\n    syntax = difflib.SequenceMatcher(None, patterns(actual), patterns(desired), autojunk=False).ratio()\\n    return {\'bucket_match\':buckets, \'move_sequence_match\':moves, \'sentence_sequence_match\':syntax,\\n            \'combined\':statistics.mean([buckets,moves,syntax])}\\n\\n\\ndef prompt_for(source, condition, source_annotation, target_structure=None):\\n    if condition not in CONDITIONS: raise ValueError(\'Unknown oracle condition\')\\n    validate_links(source_annotation[\'structure\'], source_annotation[\'semantic_ledger\'])\\n    if source_annotation[\'provenance\'][\'input_sha256\'] != sha(source): raise ValueError(\'Changed source annotation\')\\n    if condition == \'A_plain\': return make_prompt(source)\\n    structure = source_annotation[\'structure\'] if condition == \'B_source_structure\' else target_structure\\n    template = structural_template(structure)\\n    # Same source evidence in B/C. Never bind target-side IDs to source claims by position.\\n    claims = [{\'claim_id\': c[\'claim_id\'], \'source_evidence\': c[\'evidence\'][\'text\']}\\n              for c in source_annotation[\'semantic_ledger\'][\'claims\']]\\n    constraints = (\\n        \'Use the structural template as a presentation guide, not new factual evidence. \'\\n        \'The source-evidence units below must all be preserved; they may contain multiple atomic claims. \'\\n        \'Keep their actors, qualifications and semantic relationships fixed. \'\\n        \'Move-role and syntax labels are fallible observations, not permission to invent a definition, \'\\n        \'contrast, result, citation, technical term or causal connection. \'\\n        \'Ignore a template feature if realizing it would add or change a claim. \'\\n        \'Sentence slots are not claim identities: combine or split presentation only when meaning is preserved. \'\\n        \'Do not output the plan or claim IDs. Return only the rewritten paragraph.\'\\n    )\\n    return make_prompt(source) + \'\\\\n\\\\n\' + constraints + \'\\\\nSource evidence units:\\\\n\' + json.dumps(claims, ensure_ascii=False) + \'\\\\nStructural template:\\\\n\' + json.dumps(template, sort_keys=True)\\n\\n\\ndef jobs_for(validation):\\n    if not validation or any(row[\'split\']!=\'validation\' for row in validation): raise ValueError(\'Oracle requires validation only\')\\n    if len({row[\'id\'] for row in validation})!=len(validation): raise ValueError(\'Duplicate validation identity\')\\n    jobs = []\\n    for index, row in enumerate(validation):\\n        a, b = extract(row[\'plain_english\']), extract(row[\'completion\'])\\n        # Rotate A/B/C execution order to balance load/order effects; all greedy controls identical.\\n        order = CONDITIONS[index%3:] + CONDITIONS[:index%3]\\n        for condition in order:\\n            prompt = prompt_for(row[\'plain_english\'], condition, a, b[\'structure\'])\\n            jobs.append({\'id\': row[\'id\'], \'split\': \'validation\', \'condition\': condition,\\n                         \'prompt\': prompt, \'prompt_sha256\': sha(prompt), \'source_sha256\': sha(row[\'plain_english\']),\\n                         \'oracle\': condition == CONDITIONS[2]})\\n    return jobs\\n\\n\\ndef validate_outputs(jobs, outputs, allow_synthetic=False):\\n    expected = {(job[\'id\'], job[\'condition\']): job for job in jobs}\\n    if len(expected)!=len(jobs) or len(outputs)!=len(jobs): raise ValueError(\'Incomplete oracle outputs\')\\n    seen = set()\\n    for row in outputs:\\n        key = (row[\'id\'], row[\'condition\']); job = expected.get(key)\\n        if key in seen or job is None or row.get(\'prompt_sha256\')!=job[\'prompt_sha256\']:\\n            raise ValueError(\'Duplicate, changed or mismatched oracle result\')\\n        if (not isinstance(row.get(\'output\'), str) or not row[\'output\'].strip() or row.get(\'hit_output_limit\') is not False\\n                or row.get(\'synthetic\') is not allow_synthetic): raise ValueError(\'Invalid or truncated oracle output\')\\n        seen.add(key)\\n\\n\\ndef generate_jobs(jobs, generator, checkpoint=None):\\n    \\"\\"\\"The generator sees a prompt-only job, never paired targets or annotations.\\"\\"\\"\\n    results = []\\n    for job in jobs:\\n        if sha(job[\'prompt\']) != job[\'prompt_sha256\']: raise ValueError(\'Changed prompt\')\\n        response = generator(copy.deepcopy(job))\\n        result = {**response, \'id\': job[\'id\'], \'condition\': job[\'condition\'],\\n                  \'prompt_sha256\': job[\'prompt_sha256\']}\\n        results.append(result)\\n        if checkpoint: checkpoint(results)\\n        if (not isinstance(result.get(\'output\'),str) or not result[\'output\'].strip()\\n                or result.get(\'hit_output_limit\') is not False):\\n            raise ValueError(\'Empty or truncated output; partial record saved, experiment incomplete\')\\n    return results\\n\\n\\ndef score(validation, jobs, outputs, profile, synthetic=False, semantic=None, calibration=None):\\n    validate_outputs(jobs, outputs, synthetic)\\n    if any(row[\'split\']!=\'validation\' for row in validation): raise ValueError(\'Invalid scoring split\')\\n    by_id = {row[\'id\']: row for row in validation}\\n    records = []\\n    for output in outputs:\\n        row = by_id[output[\'id\']]; source, target = row[\'plain_english\'], row[\'completion\']\\n        generated = extract(output[\'output\'])\\n        actual = flatten(generated[\'structure\']); target_structure = extract(target)[\'structure\']; oracle = flatten(target_structure)\\n        source_structure = extract(source)[\'structure\']\\n        requested = source_structure if output[\'condition\']==CONDITIONS[1] else target_structure if output[\'condition\']==CONDITIONS[2] else None\\n        oracle_adherence = adherence(generated[\'structure\'],target_structure)\\n        verification = verify(source, output[\'output\'])\\n        flags = risk_flags(source, output[\'output\'])\\n        independent = (semantic or {}).get((row[\'id\'], output[\'condition\']))\\n        src_tokens, out_tokens = set(tokens(source)), set(tokens(output[\'output\']))\\n        records.append({\'id\': row[\'id\'], \'condition\': output[\'condition\'],\\n            \'source_similarity\': overlap(source, output[\'output\']),\\n            \'exact_copy\': \' \'.join(source.split())==\' \'.join(output[\'output\'].split()),\\n            \'near_copy\': overlap(source, output[\'output\'])>=.90,\\n            \'semantic_surface_flags\': flags, \'bounded_verification\': verification[\'status\'],\\n            \'semantic_consistency\': independent[\'status\'] if independent else \'unresolved\',\\n            \'independent_semantic_assessment_executed\': independent is not None,\\n            \'independent_nli\': independent,\\n            \'source_token_set_coverage\': len(src_tokens & out_tokens)/max(1,len(src_tokens)),\\n            \'output_token_set_support\': len(src_tokens & out_tokens)/max(1,len(out_tokens)),\\n            \'oracle_feature_matches\': {key: actual[key]==oracle[key] for key in oracle},\\n            \'oracle_structure_match\': oracle_adherence[\'combined\'], \'oracle_adherence_components\':oracle_adherence,\\n            \'requested_plan_match\': None if requested is None else adherence(generated[\'structure\'],requested)[\'combined\'],\\n            \'style_gain_proxy\': distance(source, profile)-distance(output[\'output\'], profile),\\n            \'output_input_token_ratio\': len(tokens(output[\'output\']))/max(1,len(tokens(source))),\\n            \'input_tokens\': output.get(\'input_tokens\'), \'output_tokens\': output.get(\'output_tokens\'),\\n            \'generation_seconds\': output.get(\'generation_seconds\')})\\n    aggregate = {}\\n    for condition in CONDITIONS:\\n        group = [row for row in records if row[\'condition\']==condition]\\n        aggregate[condition] = {\'rows\': len(group),\\n            **{key: statistics.mean(row[key] for row in group) for key in\\n               (\'source_similarity\', \'exact_copy\', \'near_copy\', \'oracle_structure_match\', \'style_gain_proxy\', \'output_input_token_ratio\')},\\n            \'rows_with_semantic_surface_flags\': sum(bool(row[\'semantic_surface_flags\']) for row in group),\\n            \'semantic_consistency_rate\': None,\\n            \'requested_plan_match\': None if condition==CONDITIONS[0] else statistics.mean(row[\'requested_plan_match\'] for row in group),\\n            \'nli_proxy_consistency_rate\': (statistics.mean(row[\'semantic_consistency\']==\'proxy_consistent\' for row in group)\\n                                            if calibration and calibration.get(\'passed\') else None),\\n            \'generation_seconds\': sum(row[\'generation_seconds\'] or 0 for row in group),\\n            \'input_tokens\': sum(row[\'input_tokens\'] or 0 for row in group),\\n            \'output_tokens\': sum(row[\'output_tokens\'] or 0 for row in group)}\\n    lookup = {(row[\'id\'], row[\'condition\']): row for row in records}\\n    paired = {}\\n    for comparison in (CONDITIONS[0], CONDITIONS[1]):\\n        changes = [lookup[(row[\'id\'],CONDITIONS[2])][\'oracle_structure_match\']-lookup[(row[\'id\'],comparison)][\'oracle_structure_match\'] for row in validation]\\n        paired[\'C_minus_\'+comparison] = {\'mean_oracle_structure_gain\': statistics.mean(changes),\\n            \'improved\': sum(value>0 for value in changes), \'worsened\': sum(value<0 for value in changes), \'tied\': sum(value==0 for value in changes)}\\n    a, b, c = (aggregate[key] for key in CONDITIONS)\\n    semantic_ready = not synthetic and bool(calibration and calibration.get(\'passed\')) and all(row[\'independent_semantic_assessment_executed\'] for row in records)\\n    support = semantic_ready and c[\'nli_proxy_consistency_rate\']>=.9 and c[\'nli_proxy_consistency_rate\']>=max(a[\'nli_proxy_consistency_rate\'],b[\'nli_proxy_consistency_rate\']) and c[\'near_copy\']<=min(a[\'near_copy\'],b[\'near_copy\']) and c[\'oracle_structure_match\']-b[\'oracle_structure_match\']>=.05 and c[\'style_gain_proxy\']>max(a[\'style_gain_proxy\'],b[\'style_gain_proxy\'])\\n    return records, {\'protocol\': PROTOCOL, \'execution_kind\': \'synthetic_wiring_check\' if synthetic else \'model_inference\',\\n        \'validation_rows\': len(validation), \'conditions\': aggregate, \'paired_differences\': paired,\\n        \'oracle_is_deployment_result\': False, \'test_rows_used\': 0, \'training_called\': False,\\n        \'oracle_supported_by_automated_proxies\': bool(support), \'semantic_claims_verified\': False,\\n        \'nli_control_results\': calibration,\\n        \'mapping_gate\': \'provisionally_supported_by_proxies\' if support else \'not_supported_by_current_proxies\' if semantic_ready else \'unresolved_semantic_assessment_or_synthetic_run\',\\n        \'limitations\': \'Adherence and style share surface proxies. Token overlap is not entailment. C uses reference-derived side information; B/C share source evidence while A is the ordinary prompt.\'}\\n", "fama_style/rhetoric/schema.py": "\\"\\"\\"Closed, content-free structural vocabulary. Claims live in a separate ledger.\\"\\"\\"\\nimport re\\n\\nVERSION = \'rhetoric_v1\'\\nEXTRACTOR = \'surface_rules_v1\'\\nROLES = [\'assertion\', \'definition\', \'reporting\', \'contrast\', \'condition\', \'conclusion\', \'unknown\']\\nORGANIZATIONS = [\'single\', \'sequence\', \'conditional\', \'contrastive\', \'reporting\', \'mixed\']\\nBINS = [\'none\', \'low\', \'high\']\\nCOUNTS = [\'one\', \'two\', \'three_plus\']\\nLENGTHS = [\'short\', \'medium\', \'long\']\\nSYNTAX = [\'simple\', \'coordinated\', \'subordinated\', \'mixed\']\\nPOSITIONS = [\'none\', \'front\', \'end\', \'embedded\', \'mixed\']\\nVOICES = [\'absent\', \'present\']\\n\\n\\ndef obj(properties):\\n    return {\'type\': \'object\', \'properties\': properties, \'required\': list(properties), \'additionalProperties\': False}\\n\\n\\ndef enum(values):\\n    return {\'type\': \'string\', \'enum\': values}\\n\\n\\ndef array(item):\\n    return {\'type\': \'array\', \'items\': item, \'minItems\': 1}\\n\\n\\nUNIT = {\'type\': \'string\', \'pattern\': r\'^s[1-9][0-9]*$\'}\\nCLAIMS = array({\'type\': \'string\', \'pattern\': r\'^c[1-9][0-9]*$\'})\\nSTRUCTURE_SCHEMA = obj({\\n    \'version\': enum([VERSION]),\\n    \'paragraph\': obj({\'organization\': enum(ORGANIZATIONS), \'sentence_count\': enum(COUNTS),\\n        \'moves\': array(obj({\'unit\': UNIT, \'role\': enum(ROLES), \'claims\': CLAIMS}))}),\\n    \'sentences\': array(obj({\'unit\': UNIT, \'claims\': CLAIMS, \'syntax\': enum(SYNTAX),\\n        \'condition_position\': enum(POSITIONS), \'voice\': enum(VOICES), \'length\': enum(LENGTHS)})),\\n    \'language\': obj({\'author_voice\': enum(VOICES), \'semicolon\': enum(BINS),\\n        \'parenthesis\': enum(BINS), \'hedging\': enum(BINS), \'citation\': enum(BINS),\\n        \'nominalization\': enum(BINS)}),\\n})\\nJSON_SCHEMA = {\'$schema\': \'https://json-schema.org/draft/2020-12/schema\',\\n               \'$id\': \'urn:fama-style:rhetoric:v1\', **STRUCTURE_SCHEMA}\\n\\n\\ndef validate(value, schema=STRUCTURE_SCHEMA):\\n    \\"\\"\\"Validate this closed subset without requiring a local JSON Schema dependency.\\"\\"\\"\\n    kind = schema[\'type\']\\n    if kind == \'object\':\\n        if not isinstance(value, dict) or set(value) != set(schema[\'properties\']):\\n            raise ValueError(\'Unexpected or missing structural fields\')\\n        for key, child in schema[\'properties\'].items():\\n            validate(value[key], child)\\n    elif kind == \'array\':\\n        if not isinstance(value, list) or len(value) < schema[\'minItems\']:\\n            raise ValueError(\'Invalid structural array\')\\n        for item in value:\\n            validate(item, schema[\'items\'])\\n    elif kind == \'string\':\\n        if not isinstance(value, str):\\n            raise ValueError(\'Invalid structural label\')\\n        if \'enum\' in schema and value not in schema[\'enum\']:\\n            raise ValueError(\'Unknown structural label\')\\n        if \'pattern\' in schema and not re.fullmatch(schema[\'pattern\'], value):\\n            raise ValueError(\'Invalid claim/unit identity\')\\n    return value\\n\\n\\ndef validate_links(structure, ledger):\\n    validate(structure)\\n    ids = {claim[\'claim_id\'] for claim in ledger[\'claims\']}\\n    units = [row[\'unit\'] for row in structure[\'sentences\']]\\n    if len(set(units)) != len(units):\\n        raise ValueError(\'Duplicate structural unit\')\\n    if [row[\'unit\'] for row in structure[\'paragraph\'][\'moves\']] != units:\\n        raise ValueError(\'Paragraph moves and sentence units disagree\')\\n    if set(ref for row in structure[\'sentences\'] for ref in row[\'claims\']) != ids:\\n        raise ValueError(\'Uncovered or invented claim IDs\')\\n    for move, sentence in zip(structure[\'paragraph\'][\'moves\'], structure[\'sentences\']):\\n        if move[\'claims\'] != sentence[\'claims\']:\\n            raise ValueError(\'Move claim binding disagrees with sentence binding\')\\n    return True\\n", "fama_style/rhetoric/semantics.py": "\\"\\"\\"Independent NLI diagnostic interface, with automatic controls and abstention.\\"\\"\\"\\nimport math\\n\\nfrom fama_style.research.claims import sentences\\n\\nNLI_MODEL = \'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli\'\\n\\n\\ndef checked_prediction(predict, premise, hypothesis):\\n    result = predict(premise, hypothesis)\\n    if result is None: return None  # Context overflow must abstain, never truncate.\\n    labels = {\'entailment\', \'neutral\', \'contradiction\'}\\n    if set(result)!=labels or any(not isinstance(p, (float,int)) or not math.isfinite(p) or not 0<=p<=1 for p in result.values()):\\n        raise ValueError(\'Invalid NLI probability mapping\')\\n    if abs(sum(result.values())-1)>.01: raise ValueError(\'Invalid NLI probability sum\')\\n    return result\\n\\n\\ndef assess(source, output, predict):\\n    pairs = [(\'whole_forward\', source, output), (\'whole_reverse\', output, source)]\\n    pairs += [(f\'source_unit_{i}\', output, text) for i,text in enumerate(sentences(source),1)]\\n    pairs += [(f\'output_unit_{i}\', source, text) for i,text in enumerate(sentences(output),1)]\\n    records = [{\'direction\': direction, \'probabilities\': checked_prediction(predict, premise, hypothesis)}\\n               for direction, premise, hypothesis in pairs]\\n    known = [row[\'probabilities\'] for row in records if row[\'probabilities\'] is not None]\\n    complete = len(known)==len(records)\\n    minimum = min((p[\'entailment\'] for p in known), default=None)\\n    maximum = max((p[\'contradiction\'] for p in known), default=None)\\n    return {\'status\': \'unknown_over_context\' if not complete else \'proxy_consistent\' if minimum>=.8 and maximum<=.1 else \'proxy_flagged\',\\n            \'minimum_entailment\': minimum, \'maximum_contradiction\': maximum, \'checks\': records,\\n            \'semantic_proof\': False, \'sentence_units_are_atomic_claims\': False}\\n\\n\\ndef calibration_controls(predict):\\n    # Authored synthetic controls; no validation or reserved-test references.\\n    equivalent = [(\'Firms report revenue.\', \'Firms report revenue.\'),\\n                  (\'If demand falls, firms reduce output.\', \'Firms reduce output if demand falls.\'),\\n                  (\'Firms report revenue. Investors read reports.\', \'Firms report revenue; investors read reports.\')]\\n    changed = [(\'Firms pay investors.\', \'Investors pay firms.\'),\\n               (\'Costs cause losses.\', \'Losses cause costs.\'),\\n               (\'If demand falls, firms reduce output.\', \'Firms reduce output.\'),\\n               (\'Some firms invest.\', \'All firms invest.\'),\\n               (\'Firms may reduce output.\', \'Firms must reduce output.\'),\\n               (\'Firms do not invest.\', \'Firms invest.\'),\\n               (\'Revenue rises by 5 percent.\', \'Revenue rises by 10 percent.\')]\\n    positives = [assess(a,b,predict)[\'status\']==\'proxy_consistent\' for a,b in equivalent]\\n    negatives = [assess(a,b,predict)[\'status\']==\'proxy_flagged\' for a,b in changed]\\n    passed = all(positives) and all(negatives)\\n    return {\'equivalent_controls\': len(positives), \'accepted_equivalents\': sum(positives),\\n            \'semantic_mutations\': len(negatives), \'rejected_mutations\': sum(negatives),\\n            \'passed\': passed, \'thresholds\': {\'entailment_minimum\': .8, \'contradiction_maximum\': .1},\\n            \'meaning\': \'Small synthetic sanity gate only; not domain calibration or verified economic entailment\',\\n            \'human_review_used\': False}\\n"}')
for name, source in files.items():
    path = runtime / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")
sys.path.insert(0, str(runtime))


In [ ]:
import importlib
oracle_runtime = importlib.import_module("fama_style.rhetoric.experiment")
train = list(Path("/kaggle/input").rglob("train.jsonl"))
validation = list(Path("/kaggle/input").rglob("validation.jsonl"))
if len(train) != 1 or len(validation) != 1:
    raise ValueError("Attach exactly the frozen corpus train.jsonl and validation.jsonl split files")
oracle_runtime.main(["--train", str(train[0]), "--validation", str(validation[0]),
      "--execute-inference", "--output-dir", "/kaggle/working/rhetorical_oracle_v1"])
